# BioPred Phase 0 -- Molecule-Level Modeling Table

This notebook converts the cleaned GABA-A activity evidence artifact from Notebook 3 into a molecule-level modeling table. The cleaned evidence artifact remains activity-record level, meaning a single molecule may appear in multiple rows across endpoints, targets, assays, or documents.

The goal of this notebook is to aggregate repeated evidence into one row per molecule, create molecule-level potency summaries and candidate activity labels, preserve relevant evidence diagnostics, and save a validated modeling-table artifact for downstream molecule-level EDA, split design, and baseline modeling.

The primary aggregation policy follows the recommendation from Notebook 3: median pX is used as the main molecule-level potency summary. The leading candidate activity label is `median_px >= 6.0`, while `median_px >= 5.5` and `median_px >= 6.5` are retained as sensitivity-analysis labels.

## Objectives

1. Load the cleaned activity evidence artifact from Notebook 3.
2. Validate the handoff artifact shape, schema, molecule count, and required fields.
3. Verify activity-record grain, molecule identifier consistency, and repeated-evidence breadth before aggregation.
4. Aggregate cleaned evidence records to one row per molecule and compute molecule-level potency summaries.
5. Generate molecule-level candidate activity labels and retain evidence-quality diagnostics.
6. Validate the completed molecule-level modeling table, including shape, grain, schema, missingness, label consistency, and diagnostic-column integrity.
7. Save the validated molecule-level modeling table artifact for downstream EDA, split-readiness analysis, and baseline modeling.


In [1]:
# list imports needed for this notebook
from pathlib import Path
import sys
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import numpy as np

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

load_dotenv(paths.PROJECT_ROOT / ".env", override=True)

# create the database engine
db_url = URL.create(
    drivername = os.getenv("BIOPRED_DB_DIALECT"),
    username=os.getenv("BIOPRED_DB_USER"),
    password=os.getenv("BIOPRED_DB_PASSWORD"),
    host=os.getenv("BIOPRED_DB_HOST"),
    port=int(os.getenv("BIOPRED_DB_PORT")),
    database=os.getenv("BIOPRED_DB_NAME")
)

engine = create_engine(db_url, pool_pre_ping=True)
schema = os.getenv("BIOPRED_DB_SCHEMA", "public")

/home/ryanm/projects/BioPred


In [2]:
# load our df_clean artifact we created from the last notebook
clean_activity_evidence_path = paths.PROCESSED_DATA_DIR / "chembl_gabaa_clean_activity_evidence.parquet"

activity_evidence_clean = pd.read_parquet(clean_activity_evidence_path)

df = activity_evidence_clean

### **Section 2: Handoff Validation from Notebook 03**

This section validates that the cleaned activity evidence artifact from `03_activity_evidence_cleaning` was loaded correctly before any molecule-level aggregation is performed.

The input artifact is expected to remain at activity-record grain, where each row represents one cleaned experimental activity measurement. Because Notebook 04 will collapse this table to one row per molecule, the upstream row count, column count, unique molecule count, and required schema must be verified before continuing.

These checks act as a lightweight data contract between Notebook 03 and Notebook 04. If the cleaned evidence artifact changes unexpectedly, the notebook should fail early rather than silently producing an incorrect molecule-level modeling table.

In [3]:
# verify values for our imported df before we begin new operations

# expected handoff values from notebook_03
EXPECTED_ROWS = 4435
EXPECTED_COLS = 27
EXPECTED_UNIQUE_MOLECULES = 1546

# run basic observations for shape and nunique
actual_rows, actual_cols = df.shape
actual_unique_molecules = df["molregno"].nunique()

# run assert statements with following print statements to verify
assert actual_rows == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS:,} rows, found {actual_rows:,}"
)

assert actual_cols == EXPECTED_COLS, (
    f"Expected {EXPECTED_COLS:,} cols, found {actual_cols:,}"
)

assert actual_unique_molecules == EXPECTED_UNIQUE_MOLECULES, (
    f"Expected {EXPECTED_UNIQUE_MOLECULES:,} unique molecules, found {actual_unique_molecules:,}"
)

print("Notebook 03 handoff shape validation passed.")
print(f"Rows: {actual_rows:,}")
print(f"Cols: {actual_cols:,}")
print(f"Unique molecules: {actual_unique_molecules:,}")




Notebook 03 handoff shape validation passed.
Rows: 4,435
Cols: 27
Unique molecules: 1,546


In [4]:
# column check
df.columns.tolist()

['activity_id',
 'target_chembl_id',
 'target_pref_name',
 'target_type',
 'organism',
 'assay_id',
 'assay_chembl_id',
 'assay_type',
 'confidence_score',
 'molregno',
 'molecule_chembl_id',
 'canonical_smiles',
 'standard_inchi_key',
 'molecule_type',
 'max_phase',
 'standard_type',
 'standard_relation',
 'standard_value',
 'standard_units',
 'pchembl_value',
 'doc_id',
 'standard_value_molar',
 'px_value',
 'active_px_5_5',
 'active_px_6_0',
 'active_px_6_5',
 'beta_3_gamma_2_tgt']

In [5]:
# we will from this take 18 cols that we need for the working df for this notebook, we don't require them all for our analysis.
# # then do another assertion check to make sure we filtered correctly.  don't assume here.


id_cols = {
    "activity_id",
    "molregno",
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_inchi_key"
}

evidence_cols = {
    "target_chembl_id",
    "assay_id",
    "doc_id",
    "standard_type"
}

potency_cols = {
    "standard_value",
    "standard_units",
    "standard_value_molar",
    "pchembl_value",
    "px_value"
}

record_label_cols = {
    "active_px_5_5",
    "active_px_6_0",
    "active_px_6_5"
}

target_scope_cols = {
    "beta_3_gamma_2_tgt"
}

required_cols = id_cols | evidence_cols | potency_cols | record_label_cols | target_scope_cols

# check for any missing cols
missing_cols = required_cols - set(df.columns)

assert not missing_cols, f"Missing required columns: {sorted(missing_cols)}"

# print output of findings and status updates
print(f"Required schema validation passed: {len(required_cols)} required columns present.")

Required schema validation passed: 18 required columns present.


In [6]:
# make working df then from the selected cols
mol_df = df[[col for col in df.columns if col in required_cols]].copy()

print(f"Working dataframe shape: {mol_df.shape}")

Working dataframe shape: (4435, 18)


### **Section 3: Activity-Record Grain and Molecule Identifier Audit**

This section confirms the grain of the working dataframe before molecule-level aggregation. At this stage, each row should still represent one cleaned activity evidence record, while molecules may appear across multiple rows due to repeated assays, endpoints, targets, or documents.

The goal is to verify that the activity-record grain is intact and that the two molecule identifiers, `molregno` and `molecule_chembl_id`, define the same molecule set. If they map one-to-one, `molregno` can be used as the primary aggregation key while `molecule_chembl_id` is retained as the human-readable ChEMBL identifier.

In [7]:
# validate mol_df grain before molecule-level aggregation
n_rows = len(mol_df)
n_activity_ids = mol_df["activity_id"].nunique()
n_molecules = mol_df["molregno"].nunique()

print(f"Rows: {n_rows:,}")
print(f"Unique activity records: {n_activity_ids:,}")
print(f"Unique molecules: {n_molecules:,}")

assert n_rows == 4435
assert n_activity_ids == n_rows, (
    "Expected one row per activity record before aggregation."
)
assert n_molecules == 1546

Rows: 4,435
Unique activity records: 4,435
Unique molecules: 1,546


In [8]:
# check to see if molregno and molecule_chembl_id map one-to-one
id_pairs = mol_df[["molregno", "molecule_chembl_id"]].drop_duplicates()

print(id_pairs.shape)   # Expecting *1546, 2)

assert len(id_pairs) == 1546
print(f"Molecule identifier consistency check passed.")

(1546, 2)
Molecule identifier consistency check passed.


`molregno` and `molecule_chembl_id` produce 1,546 unique identifier pairs, matching the expected molecule count. This supports using `molregno` as the primary aggregation key while retaining `molecule_chembl_id` as the human-readable ChEMBL molecule identifier.

In [9]:
# look at activity evidence density per molecule, how many records does each molecule have?
records_per_molecule = (
    mol_df
    .groupby("molregno")
    .agg(
        n_activity_records = ("activity_id", "nunique")
    )
    .reset_index()
)

records_per_molecule["n_activity_records"].describe()

count    1546.000000
mean        2.868693
std         3.404389
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        50.000000
Name: n_activity_records, dtype: float64

The molecule-level evidence density audit shows that molecules have a median of 2 activity records and a mean of approximately 2.87 activity records. Most molecules have limited repeated evidence, but the maximum of 50 records indicates that some molecules are represented across many measurements. This supports preserving evidence-count diagnostics during molecule-level aggregation rather than collapsing directly to potency summaries alone.

In [10]:
# compute how many molecules have a single record vs multiple records
single_record_molecules = (
    records_per_molecule["n_activity_records"].eq(1).sum()
)

multi_record_molecules = (
    records_per_molecule["n_activity_records"].gt(1).sum()
)

total_molecules = 1546

single_record_share = single_record_molecules / total_molecules
multi_record_share = multi_record_molecules / total_molecules

print(f"Single-record molecules: {single_record_molecules:,} ({single_record_share:.2%})")
print(f"Multi-record molecules: {multi_record_molecules:,} ({multi_record_share:.2%})") 

Single-record molecules: 724 (46.83%)
Multi-record molecules: 822 (53.17%)


More than half of molecules have multiple cleaned activity records, indicating that molecule-level aggregation is a meaningful modeling decision rather than a simple deduplication step. Because repeated measurements may vary across assays, targets, endpoints, or documents, downstream aggregation should preserve evidence-count and variability diagnostics alongside median potency.

In [11]:
# inspect molecules with the most repeated activity evidence
top_evidence_molecules = (
    records_per_molecule
    .sort_values("n_activity_records", ascending = False)
    .head(10)
)

top_evidence_molecules

,molregno,n_activity_records
2,721,50
59,19191,38
34,10909,32
115,49870,30
191,80965,28
287,90078,27
169,80527,27
3,729,22
47,13358,20
291,90182,19


The repeated-evidence distribution has a small upper tail: the highest-evidence molecules have 50, 38, 32, and 30 records, after which the counts decline into the 15–20 record range. This suggests that most molecules do not have extreme replication, but a small subset may carry substantial repeated assay evidence. Notebook 04 should therefore preserve both central potency summaries and evidence-variability diagnostics.

In [12]:
# looking more into the high-evidence molregno values
molecule_lookup = (
    mol_df[["molregno", "molecule_chembl_id", "canonical_smiles"]]
    .drop_duplicates()
)

top_evidence_molecules_labeled = (
    top_evidence_molecules
    .merge(
        molecule_lookup,
        on = "molregno",
        how = "left"
    )
)

top_evidence_molecules_labeled

,molregno,n_activity_records,molecule_chembl_id,canonical_smiles
0,721,50,CHEMBL12,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21
1,19191,38,CHEMBL17468,CCc1c(C(=O)OC)[nH]cc2nc3cc(OC)c(OC)cc3c1-2
2,10909,32,CHEMBL96,NCCCC(=O)O
3,49870,30,CHEMBL286594,C#Cc1ccc2c(c1)C(=O)N(C)Cc1c(C(=O)OCC)ncn1-2
4,80965,28,CHEMBL911,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1
5,90078,27,CHEMBL58757,C#Cc1ccc2c(c1)C(=O)N(C)Cc1c(C(=O)OC(C)(C)C)ncn1-2
6,80527,27,CHEMBL52030,CCOC(=O)c1ncn2c1[C@@H]1CCCN1C(=O)c1cc(OC)ccc1-2
7,729,22,CHEMBL6597,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(N=[N+]=[N-])ccc1-2
8,13358,20,CHEMBL13662,Cc1nnc2ccc(-c3cccc(C(F)(F)F)c3)nn12
9,90182,19,CHEMBL59181,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(C#C[Si](C)(C)C)...


Mapping high-evidence `molregno` values back to ChEMBL molecule identifiers and SMILES confirms that the repeated-evidence tail contains recognizable and biologically plausible GABA-A-related compounds. For example, the highest-evidence molecule maps to `CHEMBL12`, while another high-evidence entry maps to `CHEMBL96`. This supports the interpretation that high record counts likely reflect well-studied compounds rather than accidental duplication alone.

In [13]:
# inspect whether high-evidence molecules also span multiple assays/documents/targets
evidence_audit = (
    mol_df
    .groupby("molregno")
    .agg(
        n_activity_records = ("activity_id", "nunique"),
        n_assays = ("assay_id", "nunique"),
        n_documents = ("doc_id", "nunique"),
        n_targets = ("target_chembl_id", "nunique"),
        n_endpoint_types = ("standard_type", "nunique")
    )
    .reset_index()
)

evidence_audit.head()

,molregno,n_activity_records,n_assays,n_documents,n_targets,n_endpoint_types
0,634,14,14,4,8,1
1,635,14,14,5,8,1
2,721,50,50,23,9,3
3,729,22,22,7,9,1
4,730,9,9,2,5,1


The evidence-breadth audit shows that repeated molecule records can reflect broad experimental coverage rather than simple row duplication. For example, some high-evidence molecules span many assays, documents, targets, and endpoint types. These diagnostics should be retained during molecule-level aggregation so that downstream modeling and interpretation can distinguish sparsely measured molecules from heavily studied compounds.

In [14]:
# do dist summaries on the 5 cols we just used (not molregno)
evidence_cols = [
    "n_activity_records",
    "n_assays",
    "n_documents",
    "n_targets",
    "n_endpoint_types"
]

evidence_audit[evidence_cols].describe()

,n_activity_records,n_assays,n_documents,n_targets,n_endpoint_types
count,1546.000000,1546.000000,1546.000000,1546.000000,1546.000000
mean,2.868693,2.804010,1.205045,2.234799,1.056921
std,3.404389,3.346796,0.922218,1.684455,0.250554
min,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.000000,1.000000
50%,2.000000,1.000000,1.000000,1.000000,1.000000
75%,4.000000,4.000000,1.000000,4.000000,1.000000
max,50.000000,50.000000,23.000000,12.000000,3.000000


The evidence-breadth summary shows that the typical molecule has limited evidence breadth, with a median of 2 activity records, 1 assay, 1 document, 1 target, and 1 endpoint type. However, the upper tail is substantial: some molecules span up to 50 activity records, 50 assays, 23 documents, 12 targets, and 3 endpoint types. This indicates that repeated evidence is not merely duplicate inflation; for some molecules, activity evidence reflects broad experimental coverage across assay, document, target, and endpoint contexts.

In [15]:
# looking now for endpoint-type distribution at activity-record grain
endpoint_counts = (
    mol_df["standard_type"]
    .value_counts()
    .rename_axis("standard_type")
    .reset_index(name="n_activity_records")
)

endpoint_counts["record_share"] = endpoint_counts["n_activity_records"] / len(mol_df)

endpoint_counts

,standard_type,n_activity_records,record_share
0,Ki,3516,0.792785
1,IC50,460,0.103720
2,EC50,459,0.103495


Consistent with the cleaned evidence audit from Notebook 03, Ki remains the dominant endpoint type in the Notebook 04 working dataframe, accounting for approximately 79% of activity records. IC50 and EC50 each contribute approximately 10%. This check confirms the endpoint composition of the handoff artifact immediately before molecule-level aggregation.

### **Section 4: Molecule-Level Potency Aggregation**

This section aggregates the cleaned activity evidence table from activity-record grain to molecule grain. The output should contain one row per molecule, with central potency summaries, potency variability diagnostics, and retained evidence-breadth counts.

The primary molecule-level potency summary is `median_px`, following the aggregation policy defined in Notebook 3. Additional summaries such as `mean_px`, `min_px`, `max_px`, `px_range`, and `px_std` are retained to support downstream EDA, label diagnostics, and sensitivity analysis.

#### **4.1: Aggregation Policy**

Notebook 04 aggregates the cleaned activity evidence table from activity-record grain to molecule grain using `molregno` as the primary key.

The primary potency summary is `median_px`, computed from `px_value`, because repeated evidence is common and median pX is more robust to assay-level variation than mean or maximum potency. Additional potency diagnostics are retained, including `mean_px`, `min_px`, `max_px`, `px_range`, and `px_std`.

Evidence diagnostics are preserved using per-molecule counts of activity records, assays, documents, targets, and endpoint types. The leading candidate activity label will use `median_px >= 6.0`, with `median_px >= 5.5` and `median_px >= 6.5` retained for sensitivity analysis.

In [16]:
# prep cell, define our aggregation design
aggregation_key = "molregno"

identifier_cols = [
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_inchi_key"
]

potency_source_col = "px_value"

evidence_count_cols = {
    "n_activity_records" : "activity_id",
    "n_assays" : "assay_id",
    "n_documents" : "doc_id",
    "n_targets" : "target_chembl_id",
    "n_endpoint_types" : "standard_type"
}


In [17]:
# build the molecule-level aggregation table, draft to be revised.
molecule_table = (
    mol_df
    .groupby(aggregation_key, as_index = False)
    .agg(
        molecule_chembl_id = ("molecule_chembl_id", "first"),
        canonical_smiles = ("canonical_smiles", "first"),
        standard_inchi_key = ("standard_inchi_key", "first"),

        n_activity_records = (evidence_count_cols["n_activity_records"], "nunique"),
        n_assays = (evidence_count_cols["n_assays"], "nunique"),
        n_documents = (evidence_count_cols["n_documents"], "nunique"),
        n_targets = (evidence_count_cols["n_targets"], "nunique"),
        n_endpoint_types = (evidence_count_cols["n_endpoint_types"], "nunique"),

        mean_px = (potency_source_col, "mean"),
        median_px = (potency_source_col, "median"),
        min_px = (potency_source_col, "min"),
        max_px = (potency_source_col, "max"),
        px_std = (potency_source_col, "std")
    )
)

molecule_table["px_range"] = molecule_table["max_px"] - molecule_table["min_px"]

molecule_table.head(10)

,molregno,molecule_chembl_id,canonical_smiles,standard_inchi_key,n_activity_records,n_assays,n_documents,n_targets,n_endpoint_types,mean_px,median_px,min_px,max_px,px_std,px_range
0,634,CHEMBL415290,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(Cl)ccc1-2,QORLMYYHZLDYOR-UHFFFAOYSA-N,14,14,4,8,1,8.057772,8.036212,7.262807,9.070581,0.525041,1.807774
1,635,CHEMBL407,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(F)ccc1-2,OFBIFZUFASYYRE-UHFFFAOYSA-N,14,14,5,8,1,8.625220,9.045757,6.829738,9.221849,0.831036,2.392110
2,721,CHEMBL12,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE-UHFFFAOYSA-N,50,50,23,9,3,7.574179,7.853872,0.908000,8.376751,1.129105,7.468751
3,729,CHEMBL6597,CCOC(=O)c1ncn2c1CN(C)C(=O)c1cc(N=[N+]=[N-])ccc1-2,CFSOJZTUTOQNIA-UHFFFAOYSA-N,22,22,7,9,1,8.546320,8.481486,7.481486,9.568636,0.484010,2.087150
4,730,CHEMBL6421,CN1Cc2c(C(=O)OC(C)(C)C)ncn2-c2ccc(N=[N+]=[N-])...,ORJZGDWYWRKAAI-UHFFFAOYSA-N,9,9,2,5,1,8.421618,8.247184,7.655608,9.522879,0.653247,1.867271
5,821,CHEMBL6440,CN1Cc2c(C(=O)OCC3CC3)ncn2-c2ccc(Cl)cc2C1=O,DRCWBQAOFNLXIV-UHFFFAOYSA-N,10,10,2,5,1,7.451437,7.371611,6.774691,8.008774,0.448315,1.234083
6,868,CHEMBL267665,CCOC(=O)c1ncn2c1CN(C)C(=O)c1ccccc1-2,HNGRWUCKESPKRC-UHFFFAOYSA-N,4,4,1,4,1,8.994084,8.939713,8.698970,9.397940,0.292585,0.698970
7,908,CHEMBL6814,CN1Cc2c(C(=O)OC(C)(C)C)ncn2-c2ccc(N=C=S)cc2C1=O,UYPWOMKFYXODIW-UHFFFAOYSA-N,10,10,2,5,1,7.731065,7.512862,7.301030,8.602060,0.485201,1.301030
8,934,CHEMBL7290,CN1Cc2c(C(=O)OC(C)(C)C)ncn2-c2ccc([N+](=O)[O-]...,RELKUIIVWKRWEL-UHFFFAOYSA-N,10,10,2,5,1,7.763861,7.647817,7.302771,8.455932,0.416634,1.153161
9,945,CHEMBL6659,CN1Cc2c(C(=O)OC(C)(C)C)ncn2-c2ccc(I)cc2C1=O,YVPJMZCJVHJOFG-UHFFFAOYSA-N,10,10,2,5,1,8.336808,8.013228,7.950782,9.420216,0.590206,1.469434


The initial molecule-level aggregation produces one row per `molregno`, carrying forward stable molecule identifiers and collapsing repeated activity evidence into molecule-level diagnostics.

The first rows show that several molecules have substantial repeated evidence. For example, `CHEMBL12` has 50 activity records spanning 50 assays, 23 documents, 9 targets, and 3 endpoint types. Other molecules in the preview also show repeated assay and target coverage, confirming that aggregation is not simply removing duplicate rows; it is compressing heterogeneous experimental evidence into molecule-level summaries.

The potency summary columns capture both central tendency and variability. `median_px` is retained as the primary molecule-level potency estimate, while `mean_px`, `min_px`, `max_px`, `px_std`, and `px_range` preserve information about measurement spread. The wide `px_range` for some molecules, such as `CHEMBL12`, indicates that repeated measurements can vary substantially across experimental contexts, supporting the decision to retain variability diagnostics rather than only a single potency value.

Record-level activity labels from the cleaned evidence table are intentionally not carried forward directly. Molecule-level labels will be recreated from `median_px` so that the modeling target reflects the aggregated molecule-level potency signal rather than individual activity-record labels.

In [18]:
# validate the molecule-level grain after aggregation

n_molecule_rows = len(molecule_table)
n_unique_molregno = molecule_table["molregno"].nunique()

print(f"Molecule table rows: {n_molecule_rows:,}")
print(f"Unique molregno values: {n_unique_molregno:,}")

assert n_molecule_rows == 1546
assert n_unique_molregno == 1546

print(f"Molecule-level grain validation passed.")


Molecule table rows: 1,546
Unique molregno values: 1,546
Molecule-level grain validation passed.


In [19]:
# validate identifier consistency before relying on "first" aggregation

id_consistency = (
    mol_df
    .groupby("molregno")
    .agg(
        n_molecule_chembl_ids = ("molecule_chembl_id", "nunique"),
        n_canonical_smiles = ("canonical_smiles", "nunique"),
        n_standard_inchi_keys = ("standard_inchi_key", "nunique")
    )
    .reset_index()
)

id_consistency_summary = (
    id_consistency[
        [
            "n_molecule_chembl_ids",
            "n_canonical_smiles",
            "n_standard_inchi_keys"
        ]
    ]
    .describe()
)

id_consistency_summary

,n_molecule_chembl_ids,n_canonical_smiles,n_standard_inchi_keys
count,1546.0,1546.0,1546.0
mean,1.0,1.0,1.0
std,0.0,0.0,0.0
min,1.0,1.0,1.0
25%,1.0,1.0,1.0
50%,1.0,1.0,1.0
75%,1.0,1.0,1.0
max,1.0,1.0,1.0


Identifier consistency validation confirms that each `molregno` maps to exactly one `molecule_chembl_id`, one `canonical_smiles`, and one `standard_inchi_key`. This supports the use of `"first"` when carrying molecule identifiers into the aggregated molecule-level table.

In [20]:
# check whether missing px_std values are explained by single-record molecules

n_missing_px_std = (
    molecule_table["px_std"]
    .isna()
    .sum()
)

n_single_record_molecules = (
    molecule_table["n_activity_records"]
    .eq(1)
    .sum()    
)

print(f"Missing px_std values: {n_missing_px_std:,}")
print(f"Single-record molecules: {n_single_record_molecules:,}")

assert n_missing_px_std == 724, (
    "Expected missing px_std values to match the number of single-record molecules."
)

Missing px_std values: 724
Single-record molecules: 724


Missing `px_std` values are expected for molecules with only one cleaned activity record, because standard deviation is undefined for a single observation. Since these missing values exactly match the number of single-record molecules, they reflect aggregation mechanics rather than data quality problems.

For the molecule-level modeling table, missing `px_std` values will be filled with `0.0` to represent no observed potency variability. The `n_activity_records` column is retained so downstream analysis can distinguish single-record molecules from multi-record molecules with genuinely low variability.

In [21]:
# fill missing px_std values for single-record observations

molecule_table["px_std"] = molecule_table["px_std"].fillna(0.0)

n_missing_px_std_rem = molecule_table["px_std"].isna().sum()

print(f"Missing px_std values after fill: {n_missing_px_std_rem:,}")

#assert

Missing px_std values after fill: 0


In [22]:
# implement the compact potency-summary validation, another check to maintain our data consistency for the aggregaation

median_in_bounds = (
    (molecule_table["median_px"] >= molecule_table["min_px"])
    & (molecule_table["median_px"] <= molecule_table["max_px"])
).all()

mean_in_bounds = (
    (molecule_table["mean_px"] >= molecule_table["min_px"])
    & (molecule_table["mean_px"] <= molecule_table["max_px"])
).all()

range_matches = np.isclose(
    molecule_table["px_range"],
    molecule_table["max_px"] - molecule_table["min_px"]
).all()

print(f"median_px within min/max bounds: {median_in_bounds}")
print(f"mean_px within min/max bounds: {mean_in_bounds}")
print(f"px_range matches max_px - min_px: {range_matches}")

assert median_in_bounds
assert mean_in_bounds
assert range_matches


median_px within min/max bounds: True
mean_px within min/max bounds: True
px_range matches max_px - min_px: True


The molecule-level potency summaries are internally consistent: both `median_px` and `mean_px` fall within each molecule’s observed `min_px` and `max_px` bounds, and `px_range` matches `max_px - min_px`. This confirms that the aggregated potency summary columns are mathematically coherent before they are used to create molecule-level activity labels.

### **Section 5: Molecule-Level Candidate Activity Labels**

This section creates molecule-level candidate activity labels from `median_px`, the primary aggregated potency summary defined in Section 4.

Record-level labels from Notebook 3 are not carried forward directly because molecules may have multiple activity records with different pX values. Instead, labels are recreated after aggregation so that each molecule receives one label based on its molecule-level potency summary.

The leading candidate label is `median_px >= 6.0`, with `median_px >= 5.5` and `median_px >= 6.5` retained as sensitivity-analysis labels. These thresholds allow downstream notebooks to compare class balance and modeling behavior under less strict, primary, and more strict activity definitions.

In [23]:
# define molecular-level activity label thresholds

label_thresholds = {
    "active_median_px_5_5" : 5.5,
    "active_median_px_6_0" : 6.0,
    "active_median_px_6_5" : 6.5,
}

primary_label_col = "active_median_px_6_0"

assert primary_label_col in label_thresholds

In [24]:
# now for each threshold in label_thresholds, create a binary molecule-level label

for label_col, threshold in label_thresholds.items():
    molecule_table[label_col] = (
        molecule_table["median_px"] >= threshold
    ).astype(int)

molecule_table[
    [
        "molregno",
        "molecule_chembl_id",
        "median_px",
        *label_thresholds.keys()
    ]
].head(10)


,molregno,molecule_chembl_id,median_px,active_median_px_5_5,active_median_px_6_0,active_median_px_6_5
0,634,CHEMBL415290,8.036212,1,1,1
1,635,CHEMBL407,9.045757,1,1,1
2,721,CHEMBL12,7.853872,1,1,1
3,729,CHEMBL6597,8.481486,1,1,1
4,730,CHEMBL6421,8.247184,1,1,1
5,821,CHEMBL6440,7.371611,1,1,1
6,868,CHEMBL267665,8.939713,1,1,1
7,908,CHEMBL6814,7.512862,1,1,1
8,934,CHEMBL7290,7.647817,1,1,1
9,945,CHEMBL6659,8.013228,1,1,1


In [25]:
# summarize molecule-level candidate label prevalence

label_cols = label_thresholds.keys()

label_summary = (
    molecule_table[label_cols]
    .agg(["sum", "mean"])
    .T
    .rename(columns = {"sum" : "n_active", "mean" : "active_rate"})
)

label_summary["n_total"] = len(molecule_table)
label_summary["n_inactive"] = label_summary["n_total"] - label_summary["n_active"]

label_summary = label_summary[
    ["n_total", "n_active", "n_inactive", "active_rate"]
]

label_summary

,n_total,n_active,n_inactive,active_rate
active_median_px_5_5,1546,1298.0,248.0,0.839586
active_median_px_6_0,1546,1200.0,346.0,0.776197
active_median_px_6_5,1546,1045.0,501.0,0.675938


The molecule-level candidate labels are active-heavy across all thresholds. The primary label, `active_median_px_6_0`, identifies 1,200 of 1,546 molecules as active, corresponding to an active rate of approximately 77.6%. The less strict `median_px >= 5.5` label increases the active rate to approximately 84.0%, while the stricter `median_px >= 6.5` label reduces it to approximately 67.6%.

This monotonic decrease in active prevalence is expected as the potency threshold becomes more stringent. The active-heavy label distribution should be considered during downstream split design, baseline modeling, and metric selection.

In [26]:
# validate label monotonicity across molecule-level thresholds

label_mono = (
    (molecule_table["active_median_px_6_5"] <= molecule_table["active_median_px_6_0"])
    & (molecule_table["active_median_px_6_0"] <= molecule_table["active_median_px_5_5"])
).all()

print(f"Label monotonicity validation passed: {label_mono}")

Label monotonicity validation passed: True


Label monotonicity validation confirms that the candidate activity labels behave as expected across thresholds. Any molecule labeled active under the strict `median_px >= 6.5` threshold is also active under the primary `median_px >= 6.0` and looser `median_px >= 5.5` thresholds. This verifies that the molecule-level labels were generated consistently from `median_px`.

In [27]:
# detect record-level label conflicts within each molecule

record_label_conflicts = (
    mol_df
    .groupby("molregno")
    .agg(
        has_record_label_conflict_5_5 = ("active_px_5_5", lambda x: x.min() != x.max()),
        has_record_label_conflict_6_0 = ("active_px_6_0", lambda x: x.min() != x.max()),
        has_record_label_conflict_6_5 = ("active_px_6_5", lambda x: x.min() != x.max()),
    )
    .reset_index()
)

record_label_conflicts.head()

,molregno,has_record_label_conflict_5_5,has_record_label_conflict_6_0,has_record_label_conflict_6_5
0,634,False,False,False
1,635,False,False,False
2,721,True,True,True
3,729,False,False,False
4,730,False,False,False


The preview confirms that record-level label conflicts are being captured at the molecule level. Most displayed molecules have no conflict, while `molregno = 721` shows conflicts at all three candidate thresholds. This is consistent with its earlier evidence profile: high record count, multiple endpoint types, and wide pX variability.

These flags identify molecules whose underlying activity records do not agree cleanly around a given threshold. They are retained as evidence-quality diagnostics and will help distinguish stable molecule-level labels from labels built on more heterogeneous record-level evidence.

In [28]:
# now summarize record-level label conflict prevalence, building off of our record_label_conflicts data

conflict_cols = [
    "has_record_label_conflict_5_5",
    "has_record_label_conflict_6_0",
    "has_record_label_conflict_6_5",
]

conflict_summary = (
    record_label_conflicts[conflict_cols]
    .agg(["sum", "mean"])
    .T
    .rename(columns = {"sum" : "n_conflict", "mean" : "conflict_rate"})
)

conflict_summary["n_total"] = len(record_label_conflicts)

conflict_summary = conflict_summary[
    ["n_total", "n_conflict", "conflict_rate"]
]

conflict_summary

,n_total,n_conflict,conflict_rate
has_record_label_conflict_5_5,1546,56.0,0.036223
has_record_label_conflict_6_0,1546,101.0,0.065330
has_record_label_conflict_6_5,1546,183.0,0.118370


Record-level label conflicts are present but limited. At the primary `median_px >= 6.0` threshold, 101 of 1,546 molecules, or approximately 6.5%, have both active and inactive activity records at the record level. Conflict rates increase as the threshold becomes stricter, rising from approximately 3.6% at the 5.5 threshold to 11.8% at the 6.5 threshold.

These results support retaining conflict flags as evidence-quality diagnostics. The conflict flags do not replace the molecule-level labels, but they identify molecules where repeated activity evidence is less consistent around a given threshold.

In [29]:
# merge record-level conflict diagnostics into molecule-level table

molecule_table = (
    molecule_table
    .merge(
        record_label_conflicts,
        on = "molregno",
        how = "left"
    )
)

molecule_table.shape

(1546, 21)

The record-level conflict diagnostics are merged into `molecule_table` by `molregno`, preserving one row per molecule. These flags are retained as evidence-quality diagnostics for downstream EDA, split-readiness checks, and model interpretation; they are not intended as default predictive features.

### **Section 6: Final Modeling Table Validation**

This section validates the completed molecule-level modeling table before export. The table should now contain one row per molecule, stable molecule identifiers, aggregated potency summaries, evidence-breadth diagnostics, molecule-level candidate activity labels, and record-level conflict flags.

The goal is to confirm that the final artifact has the expected shape, grain, required columns, missing-value behavior, and label consistency before saving it for downstream molecule-level EDA, split-readiness analysis, and baseline modeling.

In [30]:
# validate final molecule-level table shape and grain

expected_rows = 1546
expected_cols = 21

actual_shape = molecule_table.shape

n_unique_molecules = molecule_table["molregno"].nunique()

print(f"Final molecule table shape: {actual_shape}")
print(f"Unique molecules: {n_unique_molecules}")

assert actual_shape == (expected_rows, expected_cols)
assert n_unique_molecules == n_unique_molecules

print("Final molecule-level shape and grain validation passed.")


Final molecule table shape: (1546, 21)
Unique molecules: 1546
Final molecule-level shape and grain validation passed.


In [31]:
# perform a revised schema validation check.

id_cols_final = [
    "molregno",
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_inchi_key"
]

evidence_cols_final = [
    "n_activity_records",
    "n_assays",
    "n_documents",
    "n_targets",
    "n_endpoint_types"
]

potency_cols_final = [
    "mean_px",
    "median_px",
    "min_px",
    "max_px",
    "px_std",
    "px_range"
]

label_cols_final = list(label_thresholds.keys())

conflict_cols_final = [
    "has_record_label_conflict_5_5",
    "has_record_label_conflict_6_0",
    "has_record_label_conflict_6_5",
]

required_cols_final = (
    id_cols_final
    + evidence_cols_final
    + potency_cols_final
    + label_cols_final
    + conflict_cols_final
)


# check for any missing cols
missing_final_cols = sorted(set(required_cols_final) - set(molecule_table.columns))

print(f"Required final columns: {len(required_cols_final)}")
print(f"Missing final columns: {missing_final_cols}")

assert not missing_final_cols

Required final columns: 21
Missing final columns: []


In [32]:
# Check final missingness profile

missing_counts = molecule_table.isna().sum()
unexpected_missing = missing_counts[missing_counts > 0]

display(unexpected_missing)

assert unexpected_missing.empty

print("Final missingness validation passed.")

Series([], dtype: int64)

Final missingness validation passed.


The final molecule-level modeling table contains no missing values after aggregation, label creation, conflict-flag merging, and `px_std` filling. This confirms that downstream notebooks can use the exported artifact without inheriting unresolved missingness from the molecule-level table construction step.

In [33]:
# verify final label and diagnostic columns
# use label_cols_final and conflict_cols_final from above

label_values = {
    col: sorted(molecule_table[col].unique())
    for col in label_cols_final
}

conflict_values = {
    col: sorted(molecule_table[col].unique())
    for col in conflict_cols_final
}

print("Label values:")
display(label_values)

print("Conflict flag values:")
display(conflict_values)

assert all(values == [0,1] for values in label_values.values())
assert all(values == [0,1] for values in conflict_values.values())

print("Final label and diagnostic column validation passed.")

Label values:


{'active_median_px_5_5': [np.int64(0), np.int64(1)],
 'active_median_px_6_0': [np.int64(0), np.int64(1)],
 'active_median_px_6_5': [np.int64(0), np.int64(1)]}

Conflict flag values:


{'has_record_label_conflict_5_5': [np.False_, np.True_],
 'has_record_label_conflict_6_0': [np.False_, np.True_],
 'has_record_label_conflict_6_5': [np.False_, np.True_]}

Final label and diagnostic column validation passed.


The final label and diagnostic columns pass integrity validation. Molecule-level activity labels contain only binary `0/1` values, while record-level conflict indicators contain only boolean `False/True` values. This confirms that the candidate labels and evidence-quality diagnostics are encoded consistently before artifact export.

### **Section 7: Save Modeling Table Artifact**

This section saves the validated molecule-level modeling table as a reusable downstream artifact. The saved table represents the modeling-ready molecule-grain dataset produced by Notebook 04.

After saving, the artifact is reloaded and checked against the in-memory table to confirm that the export preserved the expected shape, molecule count, schema, and key label columns.

In [34]:
# save our molecule_table data to our PROCESSED_DIR to use in downstream notebooks

molecule_modeling_table_path = (
    paths.PROCESSED_DATA_DIR / "chembl_gabaa_molecule_modeling_table.parquet"
)

molecule_table.to_parquet(molecule_modeling_table_path, index=False)

print(f"Saved molecule-level modeling to:")
print(molecule_modeling_table_path)

Saved molecule-level modeling to:
/home/ryanm/projects/BioPred/data/processed/chembl_gabaa_molecule_modeling_table.parquet


In [35]:
# run re-load and verification check

saved_molecule_table = pd.read_parquet(molecule_modeling_table_path)

print(f"Reloaded artifact shape: {saved_molecule_table.shape}")
print(f"Unique molecules: {saved_molecule_table['molregno'].nunique():,}")

assert saved_molecule_table.shape == molecule_table.shape
assert saved_molecule_table["molregno"].nunique() == molecule_table["molregno"].nunique()
assert list(saved_molecule_table.columns) == list(molecule_table.columns)

print("Saved molecule-level modeling table artifact validation passed.")

Reloaded artifact shape: (1546, 21)
Unique molecules: 1,546
Saved molecule-level modeling table artifact validation passed.


The validated molecule-level modeling table was saved successfully and reloaded from disk. The reloaded artifact matches the in-memory table shape, molecule count, and column order, confirming that the export preserved the expected modeling-table structure.

This artifact is now ready to serve as the downstream input for molecule-level EDA, split-readiness analysis, and baseline modeling.

### Notebook 04 Closeout

Notebook 04 converted the cleaned GABA-A activity evidence artifact from Notebook 03 into a molecule-level modeling table. The notebook intentionally changed the working grain from activity record to molecule, using `molregno` as the primary aggregation key and preserving stable molecule identifiers for downstream interpretation.

The primary molecule-level potency summary is `median_px`, computed from cleaned `px_value` records. Additional potency summaries and variability diagnostics were retained, including `mean_px`, `min_px`, `max_px`, `px_std`, and `px_range`. Evidence-breadth diagnostics were also preserved through molecule-level counts of activity records, assays, documents, targets, and endpoint types.

Molecule-level candidate activity labels were generated from `median_px` thresholds. The primary label is `active_median_px_6_0`, with `active_median_px_5_5` and `active_median_px_6_5` retained for threshold sensitivity analysis. Record-level label-conflict flags were also created to identify molecules whose underlying activity records disagree around each candidate threshold.

Final validation confirmed that the completed molecule-level table has the expected molecule grain, required schema, no unresolved missing values, binary activity labels, and boolean conflict diagnostics. The validated table was saved and reloaded successfully, confirming that the exported artifact preserved the expected structure.

Molecule-level diagnostics derived from activity evidence, such as conflict flags and evidence counts, are retained for audit and EDA purposes; they should be reviewed carefully before inclusion as predictive model features to avoid leakage-like behavior.

The saved molecule-level modeling table is now the primary downstream artifact for molecule-level EDA, split-readiness analysis, feature generation, and baseline modeling. The cleaned activity evidence artifact from Notebook 03 remains the upstream provenance source for record-level audit and evidence inspection.